# 11 · Confidence, calibration, and robustness

**Level:** expert foundations. **Prerequisites:** validation/test separation, logits, softmax,
cross-entropy, and a trained CNN. **Live demonstration:** approximately 45–60 minutes.

Train a classifier, calibrate its confidence on a dedicated held-out subset, and evaluate
both clean and corrupted images. You will implement temperature scaling, negative
log-likelihood (NLL), expected calibration error (ECE), and reliability diagrams.
This notebook trains its own model and runs independently of earlier notebooks.


## 1. Start with a fixed experiment plan

Calibration asks whether predicted confidence agrees with observed correctness over groups
of examples. Robustness asks how behavior changes when inputs change. They are related
but different: calibrating a model does not make it resistant to blur or noise.


In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in the project root or in notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the complete course folder; course_utils.py is required.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import torch
import course_utils as cu

DEVICE = cu.configure(seed=42)
SMOKE = cu.SMOKE  # COURSE_SMOKE=1: validation only; too small to judge model quality.
QUICK = True     # Set False for a longer learning experiment, then Restart Kernel and Run All.


In [ ]:
import json
import pandas as pd
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, random_split
from torchvision import transforms
from torchvision.transforms.functional import gaussian_blur

cu.seed_everything(42)
N_TRAIN = 128 if SMOKE else (4000 if QUICK else 20000)
N_VAL = 48 if SMOKE else (1600 if QUICK else 5000)
N_TEST = 48 if SMOKE else (1000 if QUICK else 3000)
EPOCHS = 1 if SMOKE else (3 if QUICK else 8)
N_BINS = 5 if SMOKE else 10
# Decide these stress tests now, before looking at test outcomes.
CORRUPTION_PLAN = [('gaussian_noise', 0.10), ('gaussian_noise', 0.25),
                   ('gaussian_blur', 0.75), ('gaussian_blur', 1.50)]


## 2. Reserve separate data for model selection and calibration

| Subset | Allowed use |
|---|---|
| Training | Update network parameters |
| Tuning validation | Select the model checkpoint and architecture |
| Calibration | Fit one temperature after model selection |
| Test | Final clean evaluation and the predeclared corruption study |

We split the original validation subset in two using a fixed random seed. For a real
project, use stratification or source-aware grouping where needed and reserve enough
calibration examples. A smoke run verifies execution only; its metrics are not evidence.


In [ ]:
train_ds, validation_ds, test_ds, class_names = cu.fashion_data(
    train_transform=transforms.ToTensor(), eval_transform=transforms.ToTensor(),
    train_limit=N_TRAIN, val_limit=N_VAL, test_limit=N_TEST, seed=42,
)
n_tuning = len(validation_ds) // 2
tuning_ds, calibration_ds = random_split(
    validation_ds, [n_tuning, len(validation_ds) - n_tuning],
    generator=torch.Generator().manual_seed(2026),
)
assert set(tuning_ds.indices).isdisjoint(set(calibration_ds.indices))
train_loader = cu.loader(train_ds, batch_size=64, shuffle=True, seed=42)
tuning_loader = cu.loader(tuning_ds)
calibration_loader = cu.loader(calibration_ds)
print('Train / tune / calibrate / test:', len(train_ds), len(tuning_ds), len(calibration_ds), len(test_ds))
calibration_counts = torch.bincount(torch.tensor([int(y) for _, y in calibration_ds]),
                                   minlength=len(class_names))
display(pd.DataFrame({'class': class_names, 'calibration_examples': calibration_counts.numpy()}))


## 3. Train and freeze the selected classifier

Only the tuning subset controls checkpoint selection. After this cell, the model weights
are fixed. The next optimization will change one confidence-scaling parameter, not the CNN.


In [ ]:
model = cu.SmallCNN(num_classes=len(class_names), in_channels=1).to(DEVICE)
history = cu.fit(model, train_loader, tuning_loader, EPOCHS, DEVICE, lr=1e-3)
cu.plot_history(history)
model.eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)
calibration_logits, calibration_labels = cu.predict(model, calibration_loader, DEVICE)
print('Calibration logits shape:', tuple(calibration_logits.shape))


## 4. Define NLL and expected calibration error

NLL is cross-entropy applied to the logits and true class labels: lower is better.
ECE groups examples into confidence bins and averages each bin's absolute confidence–accuracy
gap, weighted by its population. ECE depends on binning and sample size; a low value alone
does not prove reliable probabilities. Empty bins contribute zero and are omitted from plots.

We use top-label calibration: confidence is the largest predicted probability and correctness
is whether that class matches the label. This does not evaluate every class probability separately.


In [ ]:
def calibration_statistics(logits, labels, temperature=1.0, n_bins=10):
    temperature = float(temperature)
    if temperature <= 0:
        raise ValueError('Temperature must be strictly positive')
    scaled = logits.detach().float().cpu() / temperature
    labels = labels.detach().long().cpu()
    probabilities = scaled.softmax(dim=1)
    confidence, prediction = probabilities.max(dim=1)
    correct = prediction.eq(labels).float()
    edges = torch.linspace(0, 1, n_bins + 1)
    rows = []
    ece = 0.0
    for bin_index in range(n_bins):
        # Include zero in the first bin and one in the last.
        mask = ((confidence >= edges[bin_index]) if bin_index == 0
                else (confidence > edges[bin_index])) & (confidence <= edges[bin_index + 1])
        count = int(mask.sum())
        mean_confidence = float(confidence[mask].mean()) if count else float('nan')
        accuracy = float(correct[mask].mean()) if count else float('nan')
        if count:
            ece += count / len(labels) * abs(mean_confidence - accuracy)
        rows.append({'bin': bin_index, 'count': count,
                     'confidence': mean_confidence, 'accuracy': accuracy})
    metrics = {
        'accuracy': float(correct.mean()),
        'nll': float(F.cross_entropy(scaled, labels)),
        'ece': ece,
        'mean_confidence': float(confidence.mean()),
    }
    return metrics, pd.DataFrame(rows)

uncalibrated_metrics, uncalibrated_bins = calibration_statistics(
    calibration_logits, calibration_labels, n_bins=N_BINS,
)
print('Before calibration, on the calibration subset:', uncalibrated_metrics)


## 5. Draw a reliability diagram before fitting

Points on the diagonal have matching mean confidence and accuracy. Below the diagonal,
confidence exceeds observed accuracy. The companion count plot shows how many examples
support each point. Sparse bins should not drive strong conclusions.


In [ ]:
def plot_reliability(binned, title):
    populated = binned[binned['count'] > 0]
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot([0, 1], [0, 1], '--', color='gray', label='Perfect group calibration')
    axes[0].plot(populated['confidence'], populated['accuracy'], 'o-', label='Observed bins')
    axes[0].set(xlim=(0, 1), ylim=(0, 1), xlabel='Mean confidence', ylabel='Accuracy', title=title)
    axes[0].legend(fontsize=8)
    axes[1].bar(binned['bin'], binned['count'])
    axes[1].set(xlabel='Confidence-bin index', ylabel='Examples', title='Support per bin')
    plt.tight_layout()
    plt.show()

plot_reliability(uncalibrated_bins, 'Calibration subset: before scaling')


## 6. Fit a positive temperature using calibration data only

Temperature scaling replaces logits $z$ with $z/T$ before softmax. Parameterize
$T=\exp(\log T)$ so it stays positive. Optimize calibration-set NLL and retain the best
temperature observed, including the baseline $T=1$.

Here we use a bounded numerical search range for `log_temperature` to keep the classroom
optimization stable. $T>1$ softens probabilities; $T<1$ sharpens them. Since positive
scaling preserves logit ordering, the predicted class stays the same.
The method is described in [On Calibration of Modern Neural Networks](https://arxiv.org/abs/1706.04599).


In [ ]:
# Detached CPU float64 logits make this one-scalar optimization inexpensive and stable.
# clone outside inference_mode: predict() returns inference tensors, which cannot
# be saved for backward by the temperature-fitting loss.
fit_logits = calibration_logits.detach().double().cpu().clone()
fit_labels = calibration_labels.detach().long().cpu().clone()
log_temperature = nn.Parameter(torch.zeros((), dtype=torch.float64))
temperature_optimizer = torch.optim.Adam([log_temperature], lr=0.05)
best_log_temperature = log_temperature.detach().clone()
best_nll = float(F.cross_entropy(fit_logits, fit_labels))
temperature_trace = []

for step in range(20 if SMOKE else 150):
    temperature_optimizer.zero_grad()
    loss = F.cross_entropy(fit_logits / log_temperature.exp(), fit_labels)
    loss.backward()
    temperature_optimizer.step()
    with torch.no_grad():
        log_temperature.clamp_(-4.0, 4.0)
        candidate_nll = float(F.cross_entropy(fit_logits / log_temperature.exp(), fit_labels))
        temperature_trace.append(candidate_nll)
        if np.isfinite(candidate_nll) and candidate_nll < best_nll:
            best_nll = candidate_nll
            best_log_temperature = log_temperature.detach().clone()

TEMPERATURE = float(best_log_temperature.exp())
print(f'Frozen temperature: {TEMPERATURE:.4f}')
print(f'Calibration NLL before: {uncalibrated_metrics["nll"]:.4f}; after: {best_nll:.4f}')
assert TEMPERATURE > 0 and np.isfinite(TEMPERATURE)
assert torch.equal(calibration_logits.argmax(1), (calibration_logits / TEMPERATURE).argmax(1))
plt.figure(figsize=(6, 3))
plt.plot(temperature_trace)
plt.xlabel('Optimization step')
plt.ylabel('Calibration NLL')
plt.title('Fitting one temperature')
plt.tight_layout()
plt.show()


## 7. Inspect the fitted calibration subset

This is an optimization diagnostic, not an unbiased estimate of calibration performance:
the temperature was fitted on these same examples. NLL is the optimization objective;
ECE can increase even when NLL decreases.


In [ ]:
calibrated_metrics, calibrated_bins = calibration_statistics(
    calibration_logits, calibration_labels, TEMPERATURE, N_BINS,
)
display(pd.DataFrame([uncalibrated_metrics, calibrated_metrics],
                     index=['Before', 'After']).round(4))
plot_reliability(calibrated_bins, 'Calibration subset: after fitting')


## 8. Define deterministic corruption tests before final evaluation

Noise adds seeded Gaussian pixel perturbations and clips back to `[0, 1]`.
Blur applies a fixed Gaussian kernel. Each dataset index receives the same corruption
on reruns, allowing paired comparisons. We will use the temperature learned on clean
calibration data unchanged under corruption.

These synthetic tests cover two specific changes. They are neither an adversarial-attack
evaluation nor a guarantee for real camera, population, or acquisition shifts.


In [ ]:
class CorruptedImages(Dataset):
    def __init__(self, dataset, kind, severity, seed=2026):
        self.dataset = dataset
        self.kind = kind
        self.severity = float(severity)
        self.seed = seed

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        image, label = self.dataset[index]
        if self.kind == 'gaussian_noise':
            generator = torch.Generator().manual_seed(self.seed + int(index))
            noise = torch.randn(image.shape, generator=generator, dtype=image.dtype)
            corrupted = (image + self.severity * noise).clamp(0, 1)
        elif self.kind == 'gaussian_blur':
            corrupted = gaussian_blur(image, kernel_size=[7, 7], sigma=[self.severity, self.severity])
        else:
            raise ValueError(f'Unknown corruption: {self.kind}')
        return corrupted, label

print('Frozen stress-test plan:', CORRUPTION_PLAN)
print('Frozen temperature:', TEMPERATURE)


## 9. Open the clean test set once

All training, model selection, calibration, binning, and corruption choices are now fixed.
The same logits give before/after metrics, so there is no retraining or second prediction
pass needed for temperature scaling. Accuracy must agree; confidence, NLL, and ECE can change.


In [ ]:
test_logits, test_labels = cu.predict(model, cu.loader(test_ds), DEVICE)
test_before, test_bins_before = calibration_statistics(test_logits, test_labels, 1.0, N_BINS)
test_after, test_bins_after = calibration_statistics(test_logits, test_labels, TEMPERATURE, N_BINS)
assert abs(test_before['accuracy'] - test_after['accuracy']) < 1e-7
display(pd.DataFrame([test_before, test_after], index=['Uncalibrated test', 'Calibrated test']).round(4))
plot_reliability(test_bins_before, 'Clean test: before scaling')
plot_reliability(test_bins_after, 'Clean test: after scaling')


## 10. Evaluate the predeclared stress tests

Report the original and calibrated model on exactly the same corrupted examples.
Accuracy can fall under corruption while confidence stays high. Clean-data calibration
does not guarantee calibration under distribution shift. Do not choose a new temperature
based on these test outcomes.


In [ ]:
stress_rows = []
for scaling, metrics in [('raw', test_before), ('calibrated', test_after)]:
    stress_rows.append({'condition': 'clean', 'severity': 0.0, 'scaling': scaling, **metrics})
for kind, severity in CORRUPTION_PLAN:
    corrupted_ds = CorruptedImages(test_ds, kind, severity)
    corrupted_logits, corrupted_labels = cu.predict(model, cu.loader(corrupted_ds), DEVICE)
    assert torch.equal(corrupted_labels, test_labels)
    for scaling, temperature in [('raw', 1.0), ('calibrated', TEMPERATURE)]:
        metrics, _ = calibration_statistics(corrupted_logits, corrupted_labels, temperature, N_BINS)
        stress_rows.append({'condition': kind, 'severity': severity, 'scaling': scaling, **metrics})
stress_table = pd.DataFrame(stress_rows)
display(stress_table.round(4))


## 11. Show the input changes and confidence gap

Seeing the corruption alongside the measurements helps explain what the test actually covers.
The confidence–accuracy comparison is an aggregate view; consult NLL, ECE, and reliability
diagrams together instead of reducing trustworthiness to one number.


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(13, 3))
clean_image, label = test_ds[0]
axes[0].imshow(clean_image[0], cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Clean')
for ax, (kind, severity) in zip(axes[1:], CORRUPTION_PLAN):
    altered, _ = CorruptedImages(test_ds, kind, severity)[0]
    ax.imshow(altered[0], cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'{kind.replace("gaussian_", "")}\nseverity={severity}')
for ax in axes:
    ax.axis('off')
plt.suptitle('True class: ' + class_names[label])
plt.tight_layout()
plt.show()

summary = stress_table[stress_table['scaling'] == 'calibrated'].copy()
summary['label'] = summary.apply(lambda row: f'{row["condition"]} {row["severity"]:g}', axis=1)
fig, ax = plt.subplots(figsize=(10, 4))
positions = np.arange(len(summary))
ax.bar(positions - 0.18, summary['accuracy'], width=0.36, label='Accuracy')
ax.bar(positions + 0.18, summary['mean_confidence'], width=0.36, label='Mean calibrated confidence')
ax.set_xticks(positions, summary['label'], rotation=20, ha='right')
ax.set_ylim(0, 1)
ax.legend()
ax.set_title('A clean-data temperature under input corruption')
plt.tight_layout()
plt.show()


## 12. Save the experiment evidence

The metadata records the fitted temperature, bin count, and evaluation protocol. In an
application, calibration metadata belongs to the exact checkpoint and preprocessing
pipeline it was fitted for. Replacing model weights requires reassessing calibration.


In [ ]:
cu.ARTIFACTS.mkdir(parents=True, exist_ok=True)
results_path = cu.ARTIFACTS / '11_robustness_results.csv'
metadata_path = cu.ARTIFACTS / '11_calibration_metadata.json'
checkpoint_path = cu.ARTIFACTS / '11_calibrated_cnn.pt'
stress_table.to_csv(results_path, index=False)
torch.save({'model_state_dict': {name: value.detach().cpu() for name, value in model.state_dict().items()},
            'temperature': TEMPERATURE, 'class_names': class_names,
            'architecture': 'SmallCNN', 'in_channels': 1,
            'preprocessing': 'grayscale 28x28, ToTensor, no normalization'}, checkpoint_path)
metadata = {
    'temperature': TEMPERATURE, 'ece_bins': N_BINS,
    'train_examples': len(train_ds), 'tuning_examples': len(tuning_ds),
    'calibration_examples': len(calibration_ds), 'test_examples': len(test_ds),
    'seed': 42, 'validation_partition_seed': 2026, 'corruption_seed': 2026,
    'corruptions': CORRUPTION_PLAN,
    'checkpoint': checkpoint_path.name,
    'temperature_fit_data': 'calibration subset only',
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Saved:', results_path, metadata_path, checkpoint_path, sep='\n')


## 13. Exercises and expert discussion

1. Recompute ECE with 5, 10, and 20 bins on the cached test logits. Explain why estimates
   differ without selecting a bin count to make the model look better.
2. Before another experiment, reserve a separate corrupted calibration subset and compare
   clean-only versus corruption-aware temperature fitting on a fresh test set.
3. Design an abstention policy: use validation/calibration data to choose a confidence
   threshold, then report both coverage and accuracy on accepted test examples.
4. Propose a real-world shift that these noise and blur experiments do not represent.

<details><summary>Suggested solutions and discussion</summary>

For exercise 1, call `calibration_statistics(test_logits, test_labels, TEMPERATURE, bins)`
for each predeclared bin count. More bins use fewer examples per bin and can make ECE
noisier. NLL does not depend on binning. For abstention, coverage is the fraction accepted;
accuracy is computed only among accepted examples, with the zero-accepted case handled
explicitly. A threshold selected on test data leaks evaluation information. Camera changes,
different object populations, mislabeled data, and new classes need additional evaluation;
temperature scaling cannot recover information the classifier never learned.

</details>

**Demonstration checkpoint:** draw the four data subsets, fit temperature without touching
test labels, show unchanged accuracy, and explain why corruption can invalidate confidence.


## References

- [On Calibration of Modern Neural Networks](https://arxiv.org/abs/1706.04599)
- [Benchmarking Neural Network Robustness to Common Corruptions and Perturbations](https://arxiv.org/abs/1903.12261)
- [PyTorch CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)
- [Torchvision Gaussian blur](https://docs.pytorch.org/vision/stable/generated/torchvision.transforms.functional.gaussian_blur.html)

Continue to the deployment and capstone lessons to package a trained model and build a
complete image-classification demonstration with your own data.
